# 02 · Data cleaning, step by step
Runs the same `Cleaner` class as `scripts/clean_dataset.py`, one step at a time, to show the effect of each
operation. Nothing is written to disk here; the script produces the official `data/interim/` outputs.

In [1]:
import sys, pandas as pd
from IPython.display import Image, Markdown, display
sys.path.insert(0, "../scripts")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
from clean_dataset import Cleaner
cl = Cleaner()
df = cl.load()
print(df.shape)

(7354, 36)


In [2]:
steps = [cl.exact_duplicates, cl.whitespace, cl.pump_ids, cl.equipment, cl.dates, cl.numerics,
         cl.plausibility, cl.categories, cl.symptoms, cl.partial_duplicates, cl.outliers, cl.impute,
         cl.consistency]
rows = []
for step in steps:
    before_rows, before_cells = len(df), len(cl.cells)
    df = step(df)
    rows.append({"step": step.__name__, "rows after": len(df), "cells changed": len(cl.cells) - before_cells})
pd.DataFrame(rows)

,step,rows after,cells changed
0,exact_duplicates,7246,0
1,whitespace,7246,374
2,pump_ids,7246,88
3,equipment,7246,713
4,dates,7246,829
5,numerics,7246,1599
6,plausibility,7246,38
7,categories,7246,1925
8,symptoms,7246,791
9,partial_duplicates,7210,0


## Example changes for each kind of reason
Every changed cell is recorded (`cleaning_cell_changes.csv`).

In [3]:
changes = pd.DataFrame(cl.cells)
changes.groupby("reason").head(2).sort_values("reason")[["Record_ID", "column", "old", "new", "reason"]]

,Record_ID,column,old,new,reason
4763,R-001535,Maintenance_Action,no action required,No action required,action text
4762,R-005220,Maintenance_Action,REPLACE MECHANICAL SEAL,Replace mechanical seal,action text
462,R-000002,Pump_Type,Endsuction,End Suction,category spelling
463,R-001971,Pump_Type,Inline,Vertical Inline,category spelling
1175,R-000179,Installation_Date,03-Oct-2020,2020-10-03,date format
1176,R-004145,Installation_Date,21/06/2019,2019-06-21,date format
3642,R-900022,Fault_Type,NORMAL,Normal,fault label case/punctuation
3641,R-900002,Fault_Type,AIR INGRESS,Air Ingress,fault label case/punctuation
3815,R-002518,Fault_Type,Seal Leak,Mechanical Seal Leakage,fault label synonym
3816,R-900003,Fault_Type,RESTRICTED DISCHARGE,Discharge Restriction,fault label synonym


## Unit conversions in context

In [4]:
changes[changes.reason.eq('category spelling') & changes.column.eq('Fault_Severity')].head(6)

,Record_ID,column,old,new,reason
4083,R-000538,Fault_Severity,0,No Fault,category spelling
4084,R-003064,Fault_Severity,no fault,No Fault,category spelling
4085,R-000896,Fault_Severity,N/A,No Fault,category spelling
4086,R-004146,Fault_Severity,0,No Fault,category spelling
4087,R-000004,Fault_Severity,none,No Fault,category spelling
4088,R-003782,Fault_Severity,3,High,category spelling


In [5]:
conv = df[df.Converted_Columns != ""][["Record_ID", "Converted_Columns", "Bearing_Temperature_C",
          "Suction_Pressure", "Discharge_Pressure", "Flow_Rate"]]
conv.head(8)

,Record_ID,Converted_Columns,Bearing_Temperature_C,Suction_Pressure,Discharge_Pressure,Flow_Rate
4,R-000005,Discharge_Pressure,44.3,1.980000,8.859747,132.100000
19,R-000020,Suction_Pressure,80.4,1.916739,9.270000,118.600000
27,R-000028,Flow_Rate,51.5,1.810000,8.590000,130.142453
41,R-000042,Flow_Rate,51.1,2.100000,9.090000,128.552580
57,R-000058,Suction_Pressure,56.8,1.900000,8.960000,126.000000
69,R-000070,Discharge_Pressure,53.8,1.950000,7.942746,94.700000
76,R-000077,Flow_Rate,84.6,2.010000,9.280000,117.000000
84,R-000085,Discharge_Pressure,50.4,2.000000,8.780000,133.600000


## Outlier investigation — what was kept and what was treated as an error

In [6]:
cl.outlier_table

,column,pooled IQR flags,per-pump robust z>3.5,…of which in fault/unknown records,probable entry errors (nullified)
0,Vibration_mm_s,558,1222,1213,9
1,Bearing_Temperature_C,847,756,756,0
2,Seal_Temperature_C,819,743,742,0
3,Motor_Temperature_C,226,185,178,0
4,Flow_Rate,788,462,462,0
5,Suction_Pressure,171,375,375,0
6,Discharge_Pressure,1114,538,538,0
7,Motor_Current,493,447,447,0
8,Motor_Speed_RPM,0,226,223,0
9,Pressure_Fluctuation,615,605,596,9


In [7]:
df[df.Suspect_Outlier_Columns != ""][["Record_ID", "Pump_ID", "Fault_Type", "Observed_Symptoms",
                                        "Suspect_Outlier_Columns"]].head(10)

,Record_ID,Pump_ID,Fault_Type,Observed_Symptoms,Suspect_Outlier_Columns
2,R-000003,P-001,Normal,No abnormal symptoms,Pressure_Fluctuation
238,R-000239,P-002,Normal,No abnormal symptoms,Pressure_Fluctuation
239,R-000240,P-002,Normal,No abnormal symptoms,Pressure_Fluctuation
299,R-000300,P-002,Normal,No abnormal symptoms,Vibration_mm_s
668,R-000669,P-004,Normal,No abnormal symptoms,Pressure_Fluctuation
833,R-000834,P-005,Normal,No abnormal symptoms,Vibration_mm_s
1330,R-001331,P-008,Normal,No abnormal symptoms,Vibration_mm_s
1731,R-001732,P-010,Normal,No abnormal symptoms,Vibration_mm_s
2247,R-002248,P-013,Normal,No abnormal symptoms,Pressure_Fluctuation
3111,R-003112,P-018,Normal,No abnormal symptoms,Vibration_mm_s


## Operations log with justifications

In [8]:
pd.DataFrame(cl.log)

,step,column,rows_affected,action,justification
0,1 load,*,7354,Loaded raw file as text (no automatic type con...,Prevents pandas from silently turning tokens/u...
1,2 exact duplicates,*,108,Dropped exact copies (all 29 source fields ide...,An identical row with an identical Record_ID i...
2,3 whitespace,*,374,Trimmed leading/trailing spaces and collapsed ...,'Bearing Failure ' and 'Bearing Failure' must ...
3,4 identifiers,Pump_ID,88,"Normalised 'p007', 'P7', 'PUMP-007' → 'P-007'",Pump_ID is the join key to the asset register;...
4,5 equipment,Pump_Type,212,Standardised spelling/case/abbreviation to con...,Same equipment class written differently would...
...,...,...,...,...,...
98,14 consistency,action_without_maintenance_date,0,Flagged (not altered),Cross-field rule; violations are reported for ...
99,14 consistency,hours_decrease_within_pump,0,Flagged (not altered),Cross-field rule; violations are reported for ...
100,14 consistency,discharge_not_above_suction,2,Flagged (not altered),Cross-field rule; violations are reported for ...
101,14 consistency,normal_label_with_fault_severity,0,Flagged (not altered),Cross-field rule; violations are reported for ...


## Unknown labels are kept, not guessed

In [9]:
df.Fault_Type.value_counts()

Fault_Type
Normal                     4347
Bearing Failure             667
Mechanical Seal Leakage     615
Impeller Damage             271
Cavitation                  249
Shaft Misalignment          210
Discharge Restriction       166
Motor Fault                 164
Air Ingress                 162
Pump Blockage               156
Unknown                     131
Dry Running                  72
Name: count, dtype: int64

In [10]:
df.loc[df.Label_Missing_In_Source, ['Record_ID', 'Fault_Type', 'Observed_Symptoms']].head()

,Record_ID,Fault_Type,Observed_Symptoms
15,R-000016,Unknown,Visible seal leakage
26,R-000027,Unknown,High vibration; Grinding noise
167,R-000168,Unknown,High motor current
394,R-000395,Unknown,No abnormal symptoms
499,R-000500,Unknown,No abnormal symptoms
